# Imports & Config

In [ ]:
# ======================
# Core
# ======================
import os
import math
import json
import random
from pathlib import Path

# ======================
# Numerical / CV
# ======================
import numpy as np
import cv2
import matplotlib.pyplot as plt

# ======================
# Torch (for later models)
# ======================
import torch
import torchvision

# ======================
# Colab utilities
# ======================
from google.colab import files
from IPython.display import display

# ======================
# Plot settings
# ======================
plt.rcParams["figure.figsize"] = (6, 6)
plt.rcParams["image.cmap"] = "viridis"

print("✅ Imports loaded")
print(f"🔥 Torch CUDA available: {torch.cuda.is_available()}")

In [ ]:
from openai import OpenAI
from getpass import getpass

# input api key
# get from https://openai.com/api/
api_key = getpass("Enter your OpenAI API key: ")

#gpt api
gpt_client = OpenAI(api_key=api_key)

In [ ]:
import numpy as np
import torch

def clip_image_embed(pil_img, clip_model, clip_preprocess):
    """
    Converts a PIL image into a normalized CLIP image embedding.

    Steps:
    1. Preprocess the image into the format expected by the CLIP model.
    2. Disable gradient calculation for faster inference.
    3. Encode the image using CLIP’s image encoder.
    4. Normalize the resulting embedding vector to unit length.
    """
    # Preprocess image and move it to CPU
    img_in = clip_preprocess(pil_img).unsqueeze(0).to('cpu')

    with torch.no_grad():  # Disable gradients for faster computation
        img_emb = clip_model.encode_image(img_in)
        # Normalize the embedding to prevent magnitude differences from affecting similarity
        return img_emb / img_emb.norm(dim=-1, keepdim=True)


def rank_items_from_embedding(img_emb, text_emb, menu_items, item_pointers):
    """
    Ranks menu items by similarity to an image embedding using CLIP embeddings.

    Args:
        img_emb: Torch tensor representing the image embedding (1 x D).
        text_emb: Torch tensor representing all text embeddings (N x D).
        menu_items: List of menu item names.
        item_pointers: Array mapping text embeddings back to menu item indices.

    Returns:
        A tuple of (sorted_indices, probabilities) where:
        - sorted_indices: Indices of menu_items sorted by descending similarity.
        - probabilities: Normalized softmax-like probabilities per item.
    """
    # Compute cosine similarities between image and all text embeddings
    sims = (img_emb @ text_emb.T).squeeze(0)

    # Initialize array for per-item max similarities
    per_item = np.full(len(menu_items), -1e9, dtype=np.float32)

    # Move tensor to CPU and convert to numpy for easy indexing
    sims_cpu = sims.detach().float().cpu().numpy()

    # For each menu item, store the maximum similarity from its text variants
    for i in range(len(menu_items)):
        per_item[i] = sims_cpu[item_pointers == i].max()

    # Standardize scores (z-score normalization) to stabilize scaling
    z = (per_item - per_item.mean()) / (per_item.std() + 1e-6)

    # Convert standardized scores to probabilities via softmax
    probs = np.exp(z) / np.exp(z).sum()

    # Sort by descending probability
    sorted_indices = np.argsort(-probs)
    sorted_probs = probs[sorted_indices]

    # Return sorted menu indices and corresponding probabilities
    return sorted_indices, sorted_probs


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model_type = "DPT_Large"  # best quality; use DPT_Hybrid if memory issues

MIDAS = torch.hub.load("intel-isl/MiDaS", model_type)
MIDAS.to(device)
MIDAS.eval()

transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
MIDAS_TRANSFORM = transforms.dpt_transform

# Code

## Image Helpers

In [ ]:
def show_image(img, title=""):
    plt.figure()
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()

import plotly.graph_objects as go

def plot_heatmap_overlay(data, img1=None, mask=None):
    """
    Plot a semi-transparent heatmap over an optional background image.

    Args:
        data: (H, W) scalar array
        img1: optional (H, W, 3) background image
        mask: optional (H, W) mask (truthy = keep)
    """
    data = np.asarray(data)

    if data.ndim != 2:
        raise ValueError(f"`data` must be 2D. Got {data.shape}")

    H, W = data.shape

    # Validate image
    if img1 is not None:
        img1 = np.asarray(img1)
        if img1.shape[:2] != (H, W):
            raise ValueError("`img1` must match data spatial dimensions")

    # Apply mask
    if mask is not None:
        mask = np.asarray(mask)
        if mask.shape != (H, W):
            raise ValueError("`mask` must match data shape")
        data = np.where(mask.astype(bool), data, np.nan)

    fig = go.Figure()

    # Background
    if img1 is not None:
        fig.add_trace(go.Image(z=img1))

    # Heatmap
    fig.add_trace(
        go.Heatmap(
            z=data,
            colorscale="Viridis",
            opacity=0.45,
            hovertemplate="x=%{x}<br>y=%{y}<br>value=%{z:.2f}<extra></extra>",
            colorbar=dict(title="value"),
            zsmooth=False,
        )
    )

    fig.update_layout(
        title="Heatmap Overlay",
        xaxis=dict(showticklabels=False),
        yaxis=dict(showticklabels=False, autorange="reversed"),
        width=700,
        height=700,
    )

    return fig

def image_to_base64(img):
    """Convert a PIL Image to a base64 data URI for OpenAI API."""
    buffer = BytesIO()
    img.save(buffer, format="JPEG")  # or "PNG" if you prefer
    b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"

## Part 1 - Preprocessing

## Part 2 - LLM Classification

In [ ]:
import base64
from io import BytesIO
import json

from typing import List
from pydantic import BaseModel, confloat

class ClassifiedItem(BaseModel):
    id: int
    name: str
    confidence: confloat(ge=0.0, le=1.0)

class ClassificationResult(BaseModel):
    items: List[ClassifiedItem]   # [{id,name,confidence}, ...]
    explanation: str

# gpt model 1, item classification
def gpt_item_classification(pil_img, items_ranked, client, model="gpt-5-mini"):
    # convert PIL image to base64
    image_url = image_to_base64(pil_img)

    # pair items with scores (top-N already ranked upstream)
    items_payload = json.dumps(items_ranked)

    # prompt
    prompt_text = (
        "You are a food identification expert specializing in dining hall meals.\n\n"
        "You will be shown an image of a plate and a list of candidate menu items ranked "
        "by visual similarity scores (highest to lowest).\n"
        "Your task: determine which of these items are actually present on the plate.\n\n"
        "Guidelines:\n"
        "- Use both the image and the ranking scores; scores are hints, not ground truth.\n"
        "- Focus only on visible foods; ignore background objects like trays or utensils.\n"
        "- Only choose from the provided menu item list — do not invent new items.\n\n"
        "Output Format (strict JSON):\n"
        '- \"items\": an array of objects, each with fields {\"id\", \"name\", \"confidence\"}\n'
        '- \"confidence\": a float in [0,1] for each chosen item\n'
        '- \"explanation\": a brief 1–2 sentence rationale describing your reasoning.\n'
    )

    # call the gpt responses API
    response = client.responses.parse(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    { "type": "input_text", "text": prompt_text },
                    { "type": "input_text", "text": items_payload },
                    { "type": "input_image", "image_url": image_url }
                ],
            }
        ],
        text_format=ClassificationResult
    )

    return response.output_parsed.model_dump()


## Part 3 - MiDaS Volume Estimation

### Detect Plate

In [ ]:
def overlay_mask(img, mask, alpha=0.35):
    color = np.zeros_like(img)
    color[..., 1] = 255  # green overlay
    return np.where(mask[..., None] > 0,
                    (img * (1-alpha) + color * alpha).astype(np.uint8),
                    img)

def detect_plate_mask(image_rgb: np.ndarray) -> np.ndarray:
    """
    Detect the main plate as a filled ellipse mask.

    Input:
        image_rgb: HxWx3 RGB image

    Output:
        mask: HxW uint8 mask (255 = plate, 0 = background)

    Assumptions:
        - plate is fully visible
        - plate is roughly circular
        - rim has edge contrast
    """

    h, w = image_rgb.shape[:2]

    # ---------- 1. edge map ----------
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    edges = cv2.Canny(gray, 60, 160)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)

    # ---------- 2. contours ----------
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    if not contours:
        return np.zeros((h, w), dtype=np.uint8)

    img_area = h * w
    best_score = -1
    best_ellipse = None

    # ---------- 3. evaluate candidates ----------
    for cnt in contours:
        if len(cnt) < 80:
            continue

        area = cv2.contourArea(cnt)
        if area < 0.01 * img_area:
            continue

        try:
            ellipse = cv2.fitEllipse(cnt)
        except:
            continue

        (cx, cy), (a, b), angle = ellipse
        major = max(a, b)
        minor = min(a, b)

        # reject extremely skinny ellipses
        if minor / major < 0.4:
            continue

        # ---------- rim support scoring ----------
        # sample points along ellipse perimeter
        num_samples = 360
        theta = np.linspace(0, 2*np.pi, num_samples)

        cos_a = math.cos(math.radians(angle))
        sin_a = math.sin(math.radians(angle))

        rx = major / 2
        ry = minor / 2

        hits = 0
        valid = 0

        for t in theta:
            x = rx * np.cos(t)
            y = ry * np.sin(t)

            # rotate
            xr = x * cos_a - y * sin_a
            yr = x * sin_a + y * cos_a

            px = int(cx + xr)
            py = int(cy + yr)

            if 0 <= px < w and 0 <= py < h:
                valid += 1

                # small neighborhood check for robustness
                x0 = max(px - 2, 0)
                x1 = min(px + 3, w)
                y0 = max(py - 2, 0)
                y1 = min(py + 3, h)

                if np.any(edges[y0:y1, x0:x1] > 0):
                    hits += 1

        if valid == 0:
            continue

        support_ratio = hits / valid

        # slight size prior (plate should be reasonably large)
        size_score = min(1.0, major / max(h, w))

        score = support_ratio * (0.7 + 0.3 * size_score)

        if score > best_score:
            best_score = score
            best_ellipse = ellipse

    # ---------- 4. build mask ----------
    mask = np.zeros((h, w), dtype=np.uint8)

    if best_ellipse is not None:
        cv2.ellipse(mask, best_ellipse, 255, thickness=-1)

    return mask

#fig = plot_heatmap_overlay(Z_abs_mm, img1=img1, mask=mask)
#fig.show()

### Get 3D map of empty plate

In [ ]:
def _to_u8_mask(m):
    m = (m > 0).astype(np.uint8)
    return m

def ellipse_params_from_mask(plate_mask_u8):
    """
    Returns (cx, cy, plate_diameter_px, (ellipse_center, axes, angle_deg))
    where plate_diameter_px uses the major axis length (in px).
    """
    cnts, _ = cv2.findContours(plate_mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        raise RuntimeError("No contour found in plate_mask.")
    c = max(cnts, key=cv2.contourArea)
    if len(c) < 5:
        raise RuntimeError("Not enough contour points to fit ellipse.")
    (cx, cy), (w, h), angle = cv2.fitEllipse(c)
    plate_diameter_px = float(max(w, h))  # major axis
    return (float(cx), float(cy), plate_diameter_px, ((cx, cy), (w, h), angle))

def empty_plate_height_map(
    *,
    height_px: int,
    width_px: int,
    center_xy_px: tuple[float, float],  # (cx, cy)

    # plate size in the image (px)
    plate_diameter_px: float,

    # Plate geometry (mm)
    outer_diameter_mm: float,
    inner_radius_mm: float,
    lip_end_radius_mm: float,
    slope_end_radius_mm: float,

    # Heights above table (mm)
    h_inner_mm: float = 0.0,
    h_lip_mm: float | None = None,
    h_slope_end_mm: float = 0.0,

    # Profiles
    lip_profile: str = "smooth",
    slope_profile: str = "smooth",

    feather_px: float = 2.0,
    eps: float = 1e-6,
):
    """
    Returns:
        Z_rel : (H,W) float32 height map normalized to [0,1],
                where table/outside plate = 0 and highest plate point = 1.
    """

    if h_lip_mm is None:
        h_lip_mm = h_inner_mm

    # --------------------------------------------------
    # Derived metric scale
    # --------------------------------------------------
    mm_per_px = outer_diameter_mm / plate_diameter_px
    outer_radius_mm = outer_diameter_mm / 2.0

    if not (0 <= inner_radius_mm <= lip_end_radius_mm <= slope_end_radius_mm <= outer_radius_mm):
        raise ValueError("Radii must satisfy: 0 <= inner <= lip_end <= slope_end <= outer.")

    cx, cy = center_xy_px

    # --------------------------------------------------
    # Pixel grid
    # --------------------------------------------------
    ys, xs = np.indices((height_px, width_px), dtype=np.float32)
    dx_px = xs - cx
    dy_px = ys - cy
    r_mm = np.sqrt((dx_px * mm_per_px) ** 2 + (dy_px * mm_per_px) ** 2)

    # --------------------------------------------------
    # Helpers
    # --------------------------------------------------
    def smoothstep01(t):
        t = np.clip(t, 0.0, 1.0)
        return t * t * (3.0 - 2.0 * t)

    def ramp(t, mode):
        if mode == "smooth":
            return smoothstep01(t)
        if mode == "linear":
            return np.clip(t, 0.0, 1.0)
        if mode == "flat":
            return (t >= 0).astype(np.float32)
        raise ValueError("profile must be 'smooth', 'linear', or 'flat'")

    def soft_inside(r, R):
        if feather_px <= 0:
            return (r <= R).astype(np.float32)
        feather_mm = feather_px * mm_per_px
        lo = R - feather_mm / 2.0
        hi = R + feather_mm / 2.0
        t = (hi - r) / (hi - lo)
        return np.clip(t, 0.0, 1.0).astype(np.float32)

    # --------------------------------------------------
    # Soft region gates
    # --------------------------------------------------
    inside_inner = soft_inside(r_mm, inner_radius_mm)
    inside_lip_end = soft_inside(r_mm, lip_end_radius_mm)
    inside_slope_end = soft_inside(r_mm, slope_end_radius_mm)
    inside_outer = soft_inside(r_mm, outer_radius_mm)

    lip_ring = np.clip(inside_lip_end - inside_inner, 0.0, 1.0)
    slope_ring = np.clip(inside_slope_end - inside_lip_end, 0.0, 1.0)
    outer_band = np.clip(inside_outer - inside_slope_end, 0.0, 1.0)

    # --------------------------------------------------
    # Height construction (mm)
    # --------------------------------------------------
    Z_mm = np.zeros((height_px, width_px), dtype=np.float32)

    # inner flat
    Z_mm += h_inner_mm * inside_inner

    # lip transition
    if lip_end_radius_mm > inner_radius_mm:
        t = (r_mm - inner_radius_mm) / (lip_end_radius_mm - inner_radius_mm)
        s = ramp(t, lip_profile)
        Z_lip = h_inner_mm + (h_lip_mm - h_inner_mm) * s
        Z_mm += Z_lip * lip_ring

    # slope transition
    if slope_end_radius_mm > lip_end_radius_mm:
        t = (r_mm - lip_end_radius_mm) / (slope_end_radius_mm - lip_end_radius_mm)
        s = ramp(t, slope_profile)
        Z_slope = h_lip_mm + (h_slope_end_mm - h_lip_mm) * s
        Z_mm += Z_slope * slope_ring

    # outer rim band
    Z_mm += h_slope_end_mm * outer_band

    # zero outside plate (table)
    Z_mm *= inside_outer

    # --------------------------------------------------
    # Normalize to relative [0,1]
    # --------------------------------------------------
    # Table already at 0. Find the max height on the plate (use inside_outer as mask).
    zmax = float(Z_mm.max())
    if zmax <= eps:
        return np.zeros_like(Z_mm, dtype=np.float32)

    Z_rel = (Z_mm / zmax).astype(np.float32)

    # keep outside at exactly 0 (avoid tiny numeric bleed in feather region)
    Z_rel *= inside_outer

    return Z_mm, Z_rel

### Run MiDaS on Image

In [ ]:
def run_midas(midas, midas_transform, img):
  # image_rgb is HxWx3 uint8
  input_batch = midas_transform(img).to(device)

  with torch.no_grad():
      prediction = midas(input_batch)

  depth = prediction.squeeze().cpu().numpy()

  # Resize to original image size
  depth = cv2.resize(depth, (img1.shape[1], img1.shape[0]))
  return depth

### Calibrate height map to mm

In [ ]:
# calibrate height above plate

### Calculate Volume From Height Map + Mask

In [ ]:
def calculate_volume(depth_mm, plate_mask, plate_diameter_mm):
    """
    Compute volume from a depth/height map over a circular plate region.

    Assumptions:
      - depth_mm is per-pixel "height above table" (mm) OR "depth to fill" (mm) —
        i.e., it should be nonnegative for volume contribution.
      - plate_mask marks the plate region (boolean or 0..1 float soft mask).
      - plate_diameter_mm is the real plate diameter corresponding to the plate region.

    Returns:
      volume_ml: float  # 1 mL == 1 cm^3 == 1000 mm^3
      volume_mm3: float
      pixel_area_mm2: float
    """
    depth = np.asarray(depth_mm, dtype=np.float32)
    mask = np.asarray(plate_mask, dtype=np.float32)

    if depth.shape != mask.shape:
        raise ValueError(f"depth_mm and plate_mask must have the same shape, got {depth.shape} vs {mask.shape}")

    if plate_diameter_mm <= 0:
        raise ValueError("plate_diameter_mm must be > 0")

    # Clamp inputs (volume can't be negative; mask is a weight in [0,1])
    depth = np.maximum(depth, 0.0)
    mask = np.clip(mask, 0.0, 1.0)

    # Estimate per-pixel physical area by matching mask's area to the plate's real area.
    # This avoids needing mm_per_px explicitly.
    plate_area_mm2 = np.pi * (plate_diameter_mm * 0.5) ** 2
    mask_area_px = float(mask.sum())

    if mask_area_px <= 1e-6:
        return 0.0, 0.0, 0.0

    pixel_area_mm2 = plate_area_mm2 / mask_area_px

    # Volume = sum(depth_mm * area_mm2) over pixels => mm^3
    volume_mm3 = float((depth * mask).sum()) * float(pixel_area_mm2)

    # Convert mm^3 -> mL (1 mL = 1000 mm^3)
    volume_ml = volume_mm3 / 1000.0

    return volume_ml, volume_mm3, pixel_area_mm2

### Full Volume Estimation Pipeline

In [ ]:
# ---- MEASURED-FOR-A-FACT plate parameters (EDIT THESE) ----
OUTER_DIAMETER_MM = 272.5        # e.g. 260.0
INNER_RADIUS_MM = 177/2          # flat inner region radius
LIP_END_RADIUS_MM = 188/2         # radius where lip ends
SLOPE_END_RADIUS_MM = 272.5/2      # radius where slope ends
# (outer radius implied = OUTER_DIAMETER_MM/2)

# Heights above table
H_INNER_MM = 9.0                 # if you define table as 0 and plate inner surface is above it, set that here
H_LIP_MM = 13.0                   # height at lip end
H_SLOPE_END_MM = 20.0             # height at slope end / outer band

LIP_PROFILE = "smooth"
SLOPE_PROFILE = "linear"

# Feathering for mask blending (px)
FEATHER_PX = 2.0


def volume_estimator(img):
    plate_mask1 = detect_plate_mask(img)

    plate_mask_u8 = _to_u8_mask(plate_mask1)

    H, W = plate_mask_u8.shape

    # ---- Fit ellipse from plate mask to get center + diameter in pixels ----
    cx, cy, plate_diameter_px, ellipse = ellipse_params_from_mask(plate_mask_u8)

    empty_plate_mm_map, empty_plate_rel_map = empty_plate_height_map(
        height_px=H,
        width_px=W,
        center_xy_px=(cx, cy),
        plate_diameter_px=plate_diameter_px,
        outer_diameter_mm=OUTER_DIAMETER_MM,
        inner_radius_mm=INNER_RADIUS_MM,
        lip_end_radius_mm=LIP_END_RADIUS_MM,
        slope_end_radius_mm=SLOPE_END_RADIUS_MM,
        h_inner_mm=H_INNER_MM,
        h_lip_mm=H_LIP_MM,
        h_slope_end_mm=H_SLOPE_END_MM,
        lip_profile=LIP_PROFILE,
        slope_profile=SLOPE_PROFILE,
        feather_px=FEATHER_PX,
    )

    midas_depth = run_midas(MIDAS, MIDAS_TRANSFORM, img1)

    midas_adj = midas_depth - 8 - (empty_plate_rel_map*5.5)
    midas_adj = midas_adj * (midas_adj >= 7)
    midas_adj_mask = midas_adj > 0

    volume_ml = calculate_volume(midas_adj, plate_mask1, 272.5)

    return volume_ml



## Part 4 - LLM Portion Estimator

In [ ]:
from typing import List
from pydantic import BaseModel, confloat

class PortionEstimate(BaseModel):
    id: int
    name: str
    num_servings: confloat(ge=0.0, le=10.0)

class PortionEstimationResult(BaseModel):
    servings: List[PortionEstimate]
    explanation: str

# gpt model 2, portion estimation
def gpt_portion_estimation(
    pil_img,
    classification_result,
    classification_volumes_by_item,
    client,
    model="gpt-5-mini",
):
    image_url = image_to_base64(pil_img)

    payload_obj = {
        "items": classification_result["items"],
        "classification_volumes_ml": {
            str(item_id): float(volume_ml)
            for item_id, volume_ml in (classification_volumes_by_item or {}).items()
        },
    }
    payload = json.dumps(payload_obj, indent=2)

    prompt_text = (
        "You are a nutrition analyst estimating portion sizes from a single plate image.\n\n"
        "INPUTS:\n"
        "1) An image of a single plate.\n"
        "2) A JSON of detected items with nutrition info, including serving size.\n"
        "3) classification_volumes_ml: an estimated volume in mL for each detected item, keyed by item id.\n\n"
        "TASK:\n"
        "- For each detected item, output ONLY the number of servings on the plate.\n"
        "- Only return items present in the provided JSON.\n"
        "- If unsure, return your best conservative estimate.\n\n"
        "HOW TO USE VOLUME:\n"
        "- For non-EACH serving units, use the provided volume estimate heavily.\n"
        "- Convert volume (mL) to estimated weight using a reasonable food-density heuristic, then divide by the serving-size weight.\n"
        "- Use the image and item identity to choose the density. Typical heuristics:\n"
        "  * watery foods / sauces / soups / beans: about 0.9-1.1 g/mL\n"
        "  * cooked rice / pasta / mac and cheese / mashed foods: about 0.6-0.9 g/mL\n"
        "  * fries / nuggets / roasted vegetables: about 0.35-0.7 g/mL\n"
        "  * solid meats / dense mixed dishes: about 0.8-1.1 g/mL\n"
        "- If the provided volume is clearly implausible for the visible food, disregard it and rely on the image + serving size instead.\n"
        "- If the volume is plausible but somewhat noisy, use it as a strong prior, not an absolute rule.\n\n"
        "SERVING SIZE RULES:\n"
        "- If serving size uses EACH and shows a number N (example: '4 EACH'), estimate visible piece count P and report servings = P / N.\n"
        "- For EACH items, count pieces visually first. Use volume only as a weak sanity check.\n"
        "- For non-EACH items with gram or ounce serving sizes, estimate item weight from volume and density, then compute servings.\n"
        "- For cup-based serving sizes, you may reason through volume directly, but still sanity-check against likely food density and the image.\n\n"
        "OUTPUT RULES:\n"
        "- Do not invent items.\n"
        "- Do not output piece count as a field.\n"
        "- Include brief reasoning in explanation; for EACH items include the counted pieces there.\n\n"
        "FORMAT (strict JSON):\n"
        "{\n"
        '  "servings": [ { "id": <int>, "name": "<str>", "num_servings": <float> }, ... ],\n'
        '  "explanation": "<1-2 short sentences>"\n'
        "}\n"
    )

    response = client.responses.parse(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt_text},
                    {"type": "input_text", "text": payload},
                    {"type": "input_image", "image_url": image_url},
                ],
            }
        ],
        text_format=PortionEstimationResult,
    )

    return response.output_parsed.model_dump()


# Prediction Function

In [ ]:
import time
def predict(pil_image, dining_hall_id, meal, date):
  start_total = time.time()
  print(f"\n=== Running menumatch for new image ({meal}, {date}) ===")

  t0 = time.time()
  print(f"[Loaded image] {time.time() - t0:.3f}s")

  # call huskyeats api for menu items
  t0 = time.time()
  menu_items = get_menu_items(dining_hall_id, meal, date)

  menu_items_names = [item["name"] for item in menu_items]
  print(f"[Loaded menu items] {time.time() - t0:.3f}s")

  clip_prompts = [
    "a photo of {}",
    "a plate of {}",
    "{} on a plate",
    "cafeteria serving of {}",
    "dining hall style {}",
  ]

  t0 = time.time()
  # create prompt variants of each menu item
  item_variants, variant_pointers = [], []
  for idx, item in enumerate(menu_items_names):
    for p in clip_prompts:
      item_variants.append(p.format(item))
      variant_pointers.append(idx)

  variant_pointers = np.array(variant_pointers)
  print(f"[Prepared prompt variants] {time.time() - t0:.3f}s")

  t0 = time.time()
  # generate text embeddings
  # TODO: use redis cache in prod to prevent recomputation of exact same prompts
  text_tokens = tokenizer(item_variants).to('cpu')
  text_emb = model.encode_text(text_tokens)
  text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)
  print(f"[Generated text embeddings] {time.time() - t0:.3f}s")

  t0 = time.time()
  # embed image
  img_emb = clip_image_embed(pil_image, model, preprocess)
  print(f"[Generated image embedding] {time.time() - t0:.3f}s")

  t0 = time.time()
  # rank text embeddings to image embedding
  items_ranked_idxs, items_scores = rank_items_from_embedding(img_emb, text_emb, menu_items_names, variant_pointers)
  items_ranked = [
    {
        "name": menu_items[i]["name"],
        "id": menu_items[i]["id"],
        "score": float(items_scores[k])
    }
    for k, i in enumerate(items_ranked_idxs)
  ]

  print(f"[CLIP Ranking items] {time.time() - t0:.3f}s")
  print(items_ranked)
  print()

  del text_tokens, text_emb, img_emb

  t0 = time.time()
  # item classification
  classification_result = gpt_item_classification(pil_image, items_ranked, gpt_client)
  print(classification_result)
  print(f"[GPT Item classification] {time.time() - t0:.3f}s")

  # estimate volume on plate
  volume_ml = volume_estimator(pil_img)

  # use classification result and pull menu data
  t0 = time.time()
  for item in classification_result["items"]:
    item["nutrition"] = get_nutrition_info(item["id"])
  print(f"[Pulled nutrition info] {time.time() - t0:.3f}s")

  # portion estimation
  t0 = time.time()
  portion_result = gpt_portion_estimation(pil_image, classification_result, volume_ml, gpt_client)
  print(portion_result)
  print(f"[GPT Portion estimation] {time.time() - t0:.3f}s")

  print(f"=== Total runtime: {time.time() - start_total:.3f}s ===\n")
  return portion_result['servings']

# Testing